# Human Action Recognition with a Fine-Tuned CLIP Vision Transformer

**Goal:** recognise which of **15 actions** a person is doing in a photo
(calling, clapping, cycling, dancing, drinking, eating, fighting, hugging, laughing, listening to music,
running, sitting, sleeping, texting, using a laptop).

**Why a new model?** Our first model, EfficientNetB0 trained in Keras, reached **76.4%** validation accuracy.
EfficientNet was pretrained on ImageNet, which labels *objects* ("bicycle", "laptop"), not *actions*.
This notebook uses **CLIP ViT-B/16**, a Vision Transformer that OpenAI pretrained on **400 million images with
their captions** ("a man talking on the phone", "kids riding bikes"). It has already learned what human actions
look like, and we **fine-tune all of its layers** on our data.

**Plan**
1. **Data** – load labels, remove near-duplicate leakage, split into train / validation / test.
2. **Preprocessing** – resize, normalize and augment the images.
3. **Model** – CLIP ViT-B/16 with a new 15-class output layer.
4. **Training** – full fine-tuning with layer-wise learning rates, warm-up + cosine schedule,
   label smoothing and weight averaging (EMA).
5. **Evaluation** – accuracy on a held-out **test set**, per-action scores, confusion matrix.
6. **Predictions** – the model's guesses on new photos.
7. **Cross-validation** – repeat the training on 5 different splits to check that the result is reliable.

> **Before running on Kaggle:** in the right-hand panel set **Accelerator → GPU T4 x2** and **Internet → On**
> (needed to download the pretrained CLIP weights), attach the *Human Action Recognition (HAR) Dataset*
> with **Add Input**, then click **Run All**. The main training takes about **15 minutes**; the cross-validation in Part 7 adds about **35 minutes**
> (set `RUN_CROSS_VALIDATION = False` to skip it). For a long run, use **Save Version → Save & Run All**, which keeps running in the background.

*The training recipe follows the HAR-15 project ([github.com/TECHcode317/Activity](https://github.com/TECHcode317/Activity)),
which fine-tuned the same model on this dataset and reported 91.4% test accuracy.*

# Part 1 — Data

## 1.1 Import libraries
This notebook uses **PyTorch** (a deep-learning framework, like Keras/TensorFlow) and **timm**
("PyTorch Image Models"), a library with hundreds of pretrained image networks, including CLIP.

In [ ]:
import os, gc, glob, copy, math, time, random, warnings
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode
import timm
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings("ignore", message="Detected call of")   # harmless scheduler notice with mixed precision

SEED = 42
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| timm:", timm.__version__)
print("Device: ", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU - turn on the GPU accelerator in Settings!")

## 1.2 Find the dataset

In [ ]:
csv_matches = glob.glob("/kaggle/input/**/Training_set.csv", recursive=True)
assert csv_matches, "Dataset not found - click 'Add Input' and attach the Human Action Recognition dataset."
DATA_DIR = os.path.dirname(csv_matches[0])
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "train")
TEST_IMG_DIR = os.path.join(DATA_DIR, "test")
OUTPUT_DIR = "/kaggle/working"
print("Data folder:", DATA_DIR)

## 1.3 Load the labels
Each action gets a number (`calling → 0`, `clapping → 1`, …), because networks work with numbers.

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, "Training_set.csv"))
class_names = sorted(df["label"].unique())
label_to_id = {name: i for i, name in enumerate(class_names)}
df["label_id"] = df["label"].map(label_to_id)
NUM_CLASSES = len(class_names)

print(f"{len(df)} labeled images, {NUM_CLASSES} actions")
print(df["label"].value_counts().sort_index().to_string())

## 1.4 Example images

In [ ]:
sample = df.sample(8, random_state=1)
plt.figure(figsize=(14, 6))
for i, (_, row) in enumerate(sample.iterrows()):
    plt.subplot(2, 4, i + 1)
    plt.imshow(Image.open(os.path.join(TRAIN_IMG_DIR, row["filename"])))
    plt.title(row["label"])
    plt.axis("off")
plt.tight_layout()
plt.show()

## 1.5 Find near-duplicate photos (prevent data leakage)
Some photos in the dataset are near-copies of each other (the same picture saved twice, resized or recompressed).
If one copy lands in the **training** set and the other in the **test** set, the model can simply *remember*
the photo, and the test accuracy looks better than it really is. This is called **data leakage**.

To prevent it we give every photo a **fingerprint** with *difference hashing (dHash)*:
1. Shrink the photo to 9 × 8 pixels in grayscale.
2. For each pixel, record whether it is brighter than its right-hand neighbour → **64 yes/no bits**.

Photos whose fingerprints differ in **4 bits or fewer** are put in the same **group**, and a group is never
split across train, validation and test.

The fingerprint is deliberately **cautious**. Besides real copies, it also matches some *different* photos that
share a large plain background (white studio shots, black borders), because plain areas give identical bits.
Keeping two different photos on the same side of the split doesn't leak anything, so that's harmless.
To see how many matches are real copies, we also compare small 32 × 32 colour thumbnails pixel by pixel.

In [ ]:
def dhash(path):
    img = Image.open(path).convert("L").resize((9, 8), Image.BILINEAR)
    px = np.asarray(img, dtype=np.int16)
    return (px[:, 1:] > px[:, :-1]).flatten()          # 8 rows x 8 comparisons = 64 bits

paths = [os.path.join(TRAIN_IMG_DIR, f) for f in df["filename"]]
with ThreadPoolExecutor(max_workers=os.cpu_count()) as pool:
    bits = np.array(list(pool.map(dhash, paths)), dtype=np.float32)   # shape: (images, 64)

# Hamming distance = number of bits that differ between two fingerprints
pair_a, pair_b = [], []
for start in range(0, len(bits), 1024):
    chunk = bits[start:start + 1024]
    agree = chunk @ bits.T + (1 - chunk) @ (1 - bits).T
    a, b = np.nonzero(64 - agree <= 4)
    a = a + start
    keep = b > a                                        # count each pair once, skip self-matches
    pair_a.append(a[keep])
    pair_b.append(b[keep])
pair_a, pair_b = np.concatenate(pair_a), np.concatenate(pair_b)

n = len(df)
_, groups = connected_components(coo_matrix((np.ones(len(pair_a)), (pair_a, pair_b)), shape=(n, n)), directed=False)
df["group"] = groups

# How alike are the matched pairs really? Average pixel difference of 32 x 32 colour thumbnails (0-255 scale)
def thumbnail(path):
    return np.asarray(Image.open(path).convert("RGB").resize((32, 32), Image.BILINEAR), dtype=np.float32)
pixel_diff = np.array([np.abs(thumbnail(paths[a]) - thumbnail(paths[b])).mean() for a, b in zip(pair_a, pair_b)])
is_copy = pixel_diff < 10
same_label = df["label"].values[pair_a] == df["label"].values[pair_b]

print(f"Photo pairs with matching fingerprints: {len(pair_a)}")
if len(pair_a):
    print(f"  clearly copies (thumbnails differ by less than 10 out of 255): {is_copy.sum()}"
          f" - same label in {same_label[is_copy].mean() if is_copy.any() else 0:.0%} of them")
    print(f"  other matches (near-copies such as crops, and look-alikes):      {(~is_copy).sum()}")
print(f"Groups: {groups.max() + 1} groups for {n} images")

**Examples.** The first three pairs are copies of the same photo; the differences are watermarks or image quality.
The last pair is a *look-alike*: two different photos that the fingerprint matched because of their plain backgrounds.

In [ ]:
copies = np.where(is_copy)[0]
copies = copies[np.argsort(-pixel_diff[copies])][:3]      # copies with small visible differences
lookalikes = np.where(~is_copy)[0]
lookalikes = lookalikes[np.argsort(pixel_diff[lookalikes])][len(lookalikes) // 2:][:1]   # a typical look-alike
rows = [(k, "copy") for k in copies] + [(k, "look-alike") for k in lookalikes]
if rows:
    plt.figure(figsize=(8, 3.2 * len(rows)))
    for r, (k, kind) in enumerate(rows):
        for j, idx in enumerate((pair_a[k], pair_b[k])):
            plt.subplot(len(rows), 2, 2 * r + j + 1)
            plt.imshow(Image.open(paths[idx]))
            title = f"{df['filename'][idx]} · {df['label'][idx]}"
            if j == 0:
                title = f"{kind.upper()} (pixel difference {pixel_diff[k]:.1f})\n" + title
            plt.title(title, fontsize=9)
            plt.axis("off")
    plt.tight_layout()
    plt.show()

## 1.6 Split into training, validation and test sets
- **Training (80%)** – the model learns from these photos.
- **Validation (10%)** – checked after every epoch to pick the best version of the model.
- **Test (10%)** – used **only once at the very end** to report the final accuracy.

Because the validation set is used to *choose* the best epoch, its score is slightly optimistic.
The untouched **test set** gives an honest final number. *Stratified* keeps the same mix of actions in every set;
*group* keeps near-duplicates together.

(The dataset's own `test` folder has no labels, so it can't measure accuracy. Part 6 explains what we use it for.)

In [ ]:
y_all = df["label_id"].values
all_idx = np.arange(len(df))

split_test = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)
trainval_idx, test_idx = next(split_test.split(all_idx, y_all, groups))

split_val = StratifiedGroupKFold(n_splits=9, shuffle=True, random_state=SEED)
tr, va = next(split_val.split(trainval_idx, y_all[trainval_idx], groups[trainval_idx]))
train_idx, val_idx = trainval_idx[tr], trainval_idx[va]

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

assert not (set(train_df.group) & set(val_df.group) or set(train_df.group) & set(test_df.group) or set(val_df.group) & set(test_df.group))
print(f"Training:   {len(train_df):>5} images")
print(f"Validation: {len(val_df):>5} images")
print(f"Test:       {len(test_df):>5} images")
print("No near-duplicate group is shared between the sets.")

# Part 2 — Preprocessing

## 2.1 Resize, normalize and augment
**Every image (validation and test):**
- **Resize** to 224 × 224 pixels — the input size CLIP ViT-B/16 was trained on.
- **Normalize** each colour channel with CLIP's own mean and standard deviation, so the inputs look like
  what the network saw during pretraining.

**Training images only — data augmentation** (a new random version every epoch):
- **Random resized crop** — zoom into 40–100% of the photo, so the model sees people at different sizes and positions.
- **Horizontal flip** — a mirrored person is doing the same action.
- **Small rotation** (±10°) and **brightness / contrast** changes (±20%).

Augmentation makes the training set look bigger and more varied, which reduces **overfitting**.

In [ ]:
IMG_SIZE = 224
CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.4, 1.0), interpolation=InterpolationMode.BICUBIC),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10, interpolation=InterpolationMode.BILINEAR),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(CLIP_MEAN, CLIP_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(CLIP_MEAN, CLIP_STD),
])

def to_display(t):
    # Undo the normalization so a tensor can be shown as a picture
    mean, std = torch.tensor(CLIP_MEAN).view(3, 1, 1), torch.tensor(CLIP_STD).view(3, 1, 1)
    return (t * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

example = Image.open(os.path.join(TRAIN_IMG_DIR, train_df["filename"][0])).convert("RGB")
plt.figure(figsize=(14, 3.5))
plt.subplot(1, 6, 1); plt.imshow(to_display(eval_tf(example))); plt.title("original"); plt.axis("off")
for i in range(5):
    plt.subplot(1, 6, i + 2); plt.imshow(to_display(train_tf(example))); plt.title(f"augmented {i + 1}"); plt.axis("off")
plt.suptitle(f"Action: {train_df['label'][0]}")
plt.tight_layout()
plt.show()

## 2.2 Data loaders
A **Dataset** says how to load one image and its label; a **DataLoader** groups images into **batches of 32**
and loads them in parallel on the CPU while the GPU trains.

In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = os.cpu_count() or 2

class HARDataset(Dataset):
    def __init__(self, frame, img_dir, transform, labeled=True):
        self.files = frame["filename"].tolist()
        self.labels = frame["label_id"].tolist() if labeled else [-1] * len(frame)
        self.img_dir, self.transform = img_dir, transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        img = Image.open(os.path.join(self.img_dir, self.files[i])).convert("RGB")
        return self.transform(img), self.labels[i]

def make_loader(frame, img_dir, transform, train, labeled=True):
    return DataLoader(HARDataset(frame, img_dir, transform, labeled),
                      batch_size=BATCH_SIZE if train else 128, shuffle=train, drop_last=train,
                      num_workers=NUM_WORKERS, pin_memory=device.type == "cuda",
                      persistent_workers=NUM_WORKERS > 0)

train_loader = make_loader(train_df, TRAIN_IMG_DIR, train_tf, train=True)
val_loader = make_loader(val_df, TRAIN_IMG_DIR, eval_tf, train=False)
test_loader = make_loader(test_df, TRAIN_IMG_DIR, eval_tf, train=False)

x, y = next(iter(train_loader))
print("Batch of images:", tuple(x.shape), "-> 32 images, 3 colours, 224 x 224 pixels")
print("Batch of labels:", tuple(y.shape))
print("Training steps per epoch:", len(train_loader))

# Part 3 — The Model: CLIP ViT-B/16

### How a Vision Transformer (ViT) sees a photo
1. **Patches:** the 224 × 224 photo is cut into a 14 × 14 grid of **16 × 16-pixel patches** = **196 patches**.
2. **Patch embedding:** each patch becomes a list of **768 numbers** (a *token*). A special extra **[CLS] token** is added
   whose job is to collect a summary of the whole image → 197 tokens.
3. **Position embedding:** numbers added to each token so the model knows *where* each patch was.
4. **12 Transformer blocks.** Each block has:
   - **Multi-head self-attention** (12 heads): every patch looks at every other patch and decides which are relevant.
     For "calling", the patch with the **hand** can attend to the patches with the **phone** and the **ear**.
   - An **MLP** (768 → 3072 → 768) with the **GELU** activation, applied to each token.
   - **Layer normalization** and **skip connections** around both parts, which keep training stable.
5. **Head:** the final [CLS] token (768 numbers) → **dropout** → a new **Dense layer with 15 outputs** → **softmax**.

### Why CLIP pretraining?
CLIP (Contrastive Language–Image Pre-training) learned by matching **400 million internet images to their captions**.
To match "a woman talking on her phone" to the right photo, it had to learn *actions and interactions*,
not just objects. That knowledge transfers directly to our 15 actions.

In [ ]:
MODEL_NAME = "vit_base_patch16_clip_224.openai"

try:
    model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES,
                              drop_rate=0.1,        # dropout before the new output layer
                              drop_path_rate=0.1)   # stochastic depth inside the transformer
except Exception as e:
    raise RuntimeError("Could not download the pretrained CLIP weights. In Kaggle, open Settings in the right-hand "
                       "panel, turn Internet ON, and run the notebook again.") from e

model = model.to(device)
assert np.allclose(model.pretrained_cfg["mean"], CLIP_MEAN) and np.allclose(model.pretrained_cfg["std"], CLIP_STD)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model:               {MODEL_NAME}")
print(f"Parameters:          {n_params:,}")
print(f"Patches per image:   {model.patch_embed.num_patches} (+1 [CLS] token)")
print(f"Transformer blocks:  {len(model.blocks)}")
print(f"Token size:          {model.embed_dim}")
print(f"Attention heads:     {model.blocks[0].attn.num_heads}")
print(f"MLP activation:      {model.blocks[0].mlp.act}")
print(f"New output layer:    {model.head}")

# Part 4 — Training

We **fine-tune the whole network**: every one of the ~86 million weights can change, not just the output layer.
This works much better than freezing, but it must be done gently, or the pretrained knowledge is destroyed.
These are the techniques that make it gentle:

## 4.1 Layer-wise learning rates
- **Early layers** learned general things (edges, textures, body parts) → change them **very little**.
- **Later layers** learned more specific concepts → let them adapt **more**.
- The **new output layer** starts random → it learns **fastest**.

Each layer's learning rate is the base rate (0.0001) times **0.75** for every layer below the top.
The output layer gets 5 × the base rate. **Weight decay** (0.05) gently pulls weights towards zero to reduce overfitting;
it is not applied to biases, normalization layers and embeddings, where it only hurts.

In [ ]:
BASE_LR = 1e-4
LAYER_DECAY = 0.75
HEAD_LR_MULT = 5.0
WEIGHT_DECAY = 0.05

def layer_id(name, num_blocks):
    if name.startswith(("cls_token", "pos_embed", "patch_embed", "norm_pre")):
        return 0                                   # input embeddings = lowest layer
    if name.startswith("blocks."):
        return int(name.split(".")[1]) + 1         # transformer blocks 1-12
    return num_blocks + 1                          # final norm and output head = top

def build_param_groups(model):
    num_blocks = len(model.blocks)
    groups = {}
    for name, p in model.named_parameters():
        lid = layer_id(name, num_blocks)
        is_head = name.startswith("head")
        lr = BASE_LR * HEAD_LR_MULT if is_head else BASE_LR * LAYER_DECAY ** (num_blocks + 1 - lid)
        no_decay = p.ndim <= 1 or name in ("cls_token", "pos_embed")
        g = groups.setdefault((lid, is_head, no_decay), {
            "params": [], "lr": lr, "weight_decay": 0.0 if no_decay else WEIGHT_DECAY,
            "name": "output head" if is_head else ("embeddings" if lid == 0 else
                    ("final norm" if lid == num_blocks + 1 else f"block {lid}"))})
        g["params"].append(p)
    return list(groups.values())

param_groups = build_param_groups(model)
lr_table = {}
for g in param_groups:
    lr_table[g["name"]] = g["lr"]
print("Peak learning rate per layer:")
for name, lr in lr_table.items():
    print(f"  {name:<12} {lr:.2e}")

## 4.2 Optimizer, loss, schedule and weight averaging
- **Optimizer — AdamW:** Adam with *decoupled* weight decay; the standard choice for transformers.
- **Loss — cross-entropy with label smoothing 0.1:** instead of demanding 100% confidence in the correct action,
  the target is 90.7% for the correct action and spreads the rest over the others. This stops the model
  becoming over-confident, which helps on a dataset where some photos are ambiguous (calling vs. listening to music).
- **Learning-rate schedule — warm-up + cosine decay:** the rate rises from almost zero during the first epoch
  (so the random output layer can't wreck the pretrained weights), then smoothly decreases along a cosine curve
  so the model settles into a good solution.
- **Mixed precision (FP16):** the GPU computes in 16-bit numbers where it's safe → about 2× faster, less memory.
- **Gradient clipping (1.0):** caps very large updates that could destabilize training.
- **EMA (exponential moving average) of the weights:** we keep a second copy of the model whose weights are a running
  average of the training weights (99.8% old + 0.2% new after each step). The average is smoother and usually more
  accurate than the latest weights. **The EMA copy is the one we evaluate and keep.**

In [ ]:
EPOCHS = 8
WARMUP_EPOCHS = 1
LABEL_SMOOTHING = 0.1
EMA_DECAY = 0.998
GRAD_CLIP = 1.0

optimizer = torch.optim.AdamW(param_groups, betas=(0.9, 0.999))
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

steps_per_epoch = len(train_loader)
total_steps = EPOCHS * steps_per_epoch
warmup_steps = WARMUP_EPOCHS * steps_per_epoch

def lr_factor(step):
    if step < warmup_steps:
        f = step / warmup_steps                                         # linear warm-up
    else:
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        f = 0.5 * (1 + math.cos(math.pi * progress))                    # cosine decay
    return max(f, 0.01)                                                 # never below 1% of the peak

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

class EMA:
    # A running average of the model's weights
    def __init__(self, model, decay):
        self.decay = decay
        self.model = copy.deepcopy(model).eval()
        for p in self.model.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model, step):
        d = min(self.decay, (1 + step) / (10 + step))   # average over fewer steps at the very start
        for ema_v, v in zip(self.model.state_dict().values(), model.state_dict().values()):
            if ema_v.dtype.is_floating_point:
                ema_v.mul_(d).add_(v.detach(), alpha=1 - d)
            else:
                ema_v.copy_(v)

ema = EMA(model, EMA_DECAY)

steps = np.arange(total_steps)
plt.figure(figsize=(8, 3.5))
plt.plot(steps / steps_per_epoch, [BASE_LR * HEAD_LR_MULT * lr_factor(s) for s in steps], label="output head")
plt.plot(steps / steps_per_epoch, [BASE_LR * lr_factor(s) for s in steps], label="block 12 (top)")
plt.plot(steps / steps_per_epoch, [BASE_LR * LAYER_DECAY ** 12 * lr_factor(s) for s in steps], label="block 1 (bottom)")
plt.yscale("log"); plt.xlabel("Epoch"); plt.ylabel("Learning rate"); plt.title("Warm-up + cosine schedule")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 4.3 Train
Each **epoch** goes through all training photos once, in batches of 32. For every batch:
**forward pass** → **loss** → **backpropagation** (gradients) → **AdamW update** → **EMA update**.

After each epoch the **EMA model** is scored on the validation set, and the best version is saved.
Training accuracy is measured on *augmented* photos with dropout switched on, so it's normally a bit lower than validation accuracy.

In [ ]:
@torch.no_grad()
def predict(net, loader, tta=False):
    # Returns class probabilities; with tta=True also averages in the horizontally flipped image
    net.eval()
    probs, labels = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            p = net(x).float().softmax(dim=1)
            if tta:
                p = (p + net(torch.flip(x, dims=[3])).float().softmax(dim=1)) / 2
        probs.append(p.cpu())
        labels.append(y)
    return torch.cat(probs), torch.cat(labels)

def score(probs, labels):
    loss = F.nll_loss(torch.log(probs + 1e-9), labels).item()
    acc = (probs.argmax(dim=1) == labels).float().mean().item()
    return loss, acc

history = []
best_val_acc, best_state, step = -1.0, None, 0
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    t0 = time.time()
    loss_sum, correct, seen = 0.0, 0, 0
    for x, y in train_loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = model(x)                                   # forward pass
            loss = criterion(logits.float(), y)                 # how wrong are we?
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()                           # backpropagation
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP) # gradient clipping
        scaler.step(optimizer)                                  # AdamW update
        scaler.update()
        scheduler.step()                                        # next learning rate
        ema.update(model, step)                                 # update the averaged weights
        step += 1
        loss_sum += loss.item() * len(y)
        correct += (logits.argmax(dim=1) == y).sum().item()
        seen += len(y)

    val_loss, val_acc = score(*predict(ema.model, val_loader))
    history.append({"epoch": epoch, "train_loss": loss_sum / seen, "train_acc": correct / seen,
                    "val_loss": val_loss, "val_acc": val_acc})
    marker = ""
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.detach().cpu().clone() for k, v in ema.model.state_dict().items()}
        marker = "  <- best so far"
    print(f"Epoch {epoch}/{EPOCHS} | train loss {loss_sum / seen:.3f}, acc {correct / seen:.1%} | "
          f"val loss {val_loss:.3f}, acc {val_acc:.1%} | {time.time() - t0:.0f}s{marker}")

print(f"\nTraining finished in {(time.time() - start_time) / 60:.1f} minutes. Best validation accuracy: {best_val_acc:.1%}")

final_model = ema.model
final_model.load_state_dict(best_state)
torch.save(final_model.state_dict(), os.path.join(OUTPUT_DIR, "har_clip_vit_b16.pth"))
print("Saved the best model to", os.path.join(OUTPUT_DIR, "har_clip_vit_b16.pth"))

# Part 5 — Evaluation

## 5.1 Training curves
- Accuracy should rise and loss should fall, for both training and validation.
- If training accuracy kept rising while validation accuracy dropped, the model would be **overfitting**.
- Training loss can't go below about 0.55 because of label smoothing; validation loss uses plain cross-entropy.

In [ ]:
h = pd.DataFrame(history)
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(h.epoch, h.train_acc, "o-", label="Training (augmented)")
plt.plot(h.epoch, h.val_acc, "o-", label="Validation")
plt.title("Accuracy"); plt.xlabel("Epoch"); plt.legend(); plt.grid(alpha=0.3)
plt.subplot(1, 2, 2)
plt.plot(h.epoch, h.train_loss, "o-", label="Training (label-smoothed)")
plt.plot(h.epoch, h.val_loss, "o-", label="Validation")
plt.title("Loss"); plt.xlabel("Epoch"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
h.round(4)

## 5.2 Final accuracy on the test set
The test photos were never used for training or for choosing the best epoch, so this is the honest result.

**Test-time augmentation (TTA):** each test photo is classified twice, as-is and mirrored, and the two
predictions are averaged. This usually adds a little accuracy for free.

In [ ]:
test_probs, test_labels = predict(final_model, test_loader, tta=True)
test_loss, test_acc = score(test_probs, test_labels)
_, test_acc_no_tta = score(*predict(final_model, test_loader, tta=False))

print(f"Test accuracy (with flip TTA): {test_acc:.1%}")
print(f"Test accuracy (no TTA):        {test_acc_no_tta:.1%}")
print(f"Test loss:                     {test_loss:.3f}")
print(f"Best validation accuracy:      {best_val_acc:.1%}")
print()
print("Comparison")
print(f"  Random guessing                    {1 / NUM_CLASSES:6.1%}")
print(f"  EfficientNetB0 (our first model)    76.4%  (validation)")
print(f"  CLIP ViT-B/16, fine-tuned          {test_acc:6.1%}  (test)")

## 5.3 Results per action
- **Precision** – when the model says "running", how often is it right?
- **Recall** – of all real "running" photos, how many did it find?
- **F1-score** – the balance of the two.

In [ ]:
y_true = test_labels.numpy()
y_pred = test_probs.argmax(dim=1).numpy()
print(classification_report(y_true, y_pred, target_names=class_names, digits=3, zero_division=0))

## 5.4 Confusion matrix
Rows are the **true** action, columns the **predicted** action. Numbers on the diagonal are correct;
everything else is a mistake.

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(class_names, rotation=60, ha="right")
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(class_names)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8,
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xlabel("Predicted action"); ax.set_ylabel("True action")
ax.set_title(f"Confusion matrix (test set, accuracy {test_acc:.1%})")
fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

mistakes = sorted(((cm[i, j], class_names[i], class_names[j])
                   for i in range(NUM_CLASSES) for j in range(NUM_CLASSES) if i != j), reverse=True)
print("Most common mix-ups:")
for count, true_name, pred_name in mistakes[:5]:
    print(f"  {true_name:>20} predicted as {pred_name:<20} {count} times")

# Part 6 — Predictions on New Photos
The dataset comes with **5,400 extra photos without labels** (the `test` folder). It was first released for a
competition, DPhi's *Data Sprint #76: Human Activity Recognition*. Participants submitted predictions for these
photos, and the organisers kept the correct answers secret to score them. That's why there are no labels,
and why our accuracy comes from our own labeled test set in Part 5 instead.

We use the unlabeled photos for two things:
1. **A demo:** 12 random photos the model has never seen, with its guess and confidence written underneath each one.
   There's no answer key, so you judge by eye whether the guesses look right.
2. **A competition-style file:** the predicted action for all 5,400 photos is saved to `test_predictions.csv`,
   in the same order as `Testing_set.csv`, which is the format the competition asked for.

This part doesn't change the model or its accuracy.

In [ ]:
kaggle_test_df = pd.read_csv(os.path.join(DATA_DIR, "Testing_set.csv"))
kaggle_test_loader = make_loader(kaggle_test_df, TEST_IMG_DIR, eval_tf, train=False, labeled=False)
kaggle_probs, _ = predict(final_model, kaggle_test_loader, tta=True)
kaggle_test_df["label"] = [class_names[i] for i in kaggle_probs.argmax(dim=1).numpy()]
kaggle_test_df["confidence"] = kaggle_probs.max(dim=1).values.numpy().round(3)
kaggle_test_df.to_csv(os.path.join(OUTPUT_DIR, "test_predictions.csv"), index=False)
print("Saved predictions for all", len(kaggle_test_df), "test photos to", os.path.join(OUTPUT_DIR, "test_predictions.csv"))

# A plain-English description for each action
DESCRIPTIONS = {
    "calling": "Talking on the phone", "clapping": "Clapping", "cycling": "Riding a bicycle",
    "dancing": "Dancing", "drinking": "Drinking", "eating": "Eating", "fighting": "Fighting",
    "hugging": "Hugging", "laughing": "Laughing", "listening_to_music": "Listening to music",
    "running": "Running", "sitting": "Sitting", "sleeping": "Sleeping",
    "texting": "Texting on a phone", "using_laptop": "Using a laptop",
}

def show_prediction(ax, img, probs):
    # Draws the photo with the prediction written underneath it
    best, second = probs.argsort(descending=True)[:2].tolist()
    ax.imshow(img)
    ax.axis("off")
    lines = [(DESCRIPTIONS[class_names[best]], "bold", 13),
             (f"{class_names[best]} · {probs[best]:.0%} confident", "normal", 11)]
    if probs[best] < 0.70:   # not sure: also show the second guess
        lines.append((f"2nd guess: {class_names[second]} ({probs[second]:.0%})", "normal", 10))
    for k, (text, weight, size) in enumerate(lines):
        ax.annotate(text, xy=(0.5, 0), xycoords="axes fraction", xytext=(0, -8 - 18 * k),
                    textcoords="offset points", ha="center", va="top", fontsize=size, fontweight=weight)

demo = kaggle_test_df.sample(12, random_state=7)
fig, axes = plt.subplots(3, 4, figsize=(16, 15))
for ax, (i, row) in zip(axes.flat, demo.iterrows()):
    show_prediction(ax, Image.open(os.path.join(TEST_IMG_DIR, row["filename"])).convert("RGB"), kaggle_probs[i])
plt.subplots_adjust(hspace=0.5, wspace=0.1)
plt.show()

## 6.1 Try the model on any photo
`predict_photo` shows one photo with the model's prediction underneath. It works with a file path or a web address:
right-click a photo on a website, choose **Copy image address**, and paste it between the quotes.
Kaggle's **Internet** setting must be on for web addresses.

The model only knows these 15 actions, so it always picks one of them, even for a photo of something else,
such as a dog. A low confidence and a second guess are the hint that it isn't sure.

In [ ]:
import io, urllib.request

def predict_photo(source):
    # source: a web address of a photo (https://...) or a file path
    if str(source).startswith("http"):
        request = urllib.request.Request(source, headers={"User-Agent": "Mozilla/5.0"})
        img = Image.open(io.BytesIO(urllib.request.urlopen(request, timeout=20).read())).convert("RGB")
    else:
        img = Image.open(source).convert("RGB")
    x = eval_tf(img).unsqueeze(0).to(device)
    final_model.eval()
    with torch.no_grad(), torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
        probs = (final_model(x).float().softmax(dim=1) + final_model(torch.flip(x, dims=[3])).float().softmax(dim=1)) / 2
    fig, ax = plt.subplots(figsize=(6, 6))
    show_prediction(ax, img, probs[0].cpu())
    plt.show()

# Try it on any photo. Right-click a photo on a website, choose "Copy image address",
# and paste it between the quotes instead of the file path, e.g. predict_photo("https://...")
predict_photo(os.path.join(TEST_IMG_DIR, "Image_1.jpg"))

# Part 7 — 5-Fold Cross-Validation

One train/test split can be lucky or unlucky: the accuracy in Part 5 depends partly on *which* 1,249 photos
ended up in the test set. **Cross-validation** checks this by repeating the whole experiment on different splits:

1. Split all 12,600 labeled photos into **5 folds** of about 2,520 photos each
   (stratified, and near-duplicate groups kept together, as in Part 1).
2. For each fold: train a **fresh** CLIP ViT-B/16 with the same recipe on the other 4 folds, then test it on this fold.
3. Report the **mean ± standard deviation** of the 5 test accuracies.

Every photo is used for testing exactly once, and no model is ever tested on photos it trained on.

**Settings.** Each fold trains for **4 epochs** instead of 8, to keep the run time reasonable: about 7 minutes per fold,
35 minutes in total on a T4 GPU. In the main run, validation accuracy had already reached 90.6% after epoch 4 and levelled
off after epoch 5. Inside a fold there is no best-epoch selection: the final EMA weights are tested, so the test fold is never
used to make any choice.

Set `RUN_CROSS_VALIDATION = False` to skip this part.

## 7.1 Train and test one model per fold

In [ ]:
RUN_CROSS_VALIDATION = True
CV_FOLDS = 5
CV_EPOCHS = 4

def train_fresh_model(train_frame, epochs, seed):
    # Trains a new CLIP ViT-B/16 with the same recipe as Part 4 and returns its EMA copy
    seed_everything(seed)
    net = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES,
                            drop_rate=0.1, drop_path_rate=0.1).to(device)
    loader = make_loader(train_frame, TRAIN_IMG_DIR, train_tf, train=True)
    opt = torch.optim.AdamW(build_param_groups(net), betas=(0.9, 0.999))
    total, warm = epochs * len(loader), len(loader)          # 1 warm-up epoch, then cosine decay
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: max(
        s / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))), 0.01))
    grad_scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    averaged = EMA(net, EMA_DECAY)
    step = 0
    for epoch in range(epochs):
        net.train()
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                loss = criterion(net(x).float(), y)
            opt.zero_grad(set_to_none=True)
            grad_scaler.scale(loss).backward()
            grad_scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(net.parameters(), GRAD_CLIP)
            grad_scaler.step(opt)
            grad_scaler.update()
            sched.step()
            averaged.update(net, step)
            step += 1
    return averaged.model

cv_results = []
if RUN_CROSS_VALIDATION:
    folds = StratifiedGroupKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
    cv_start = time.time()
    for fold, (tr_idx, te_idx) in enumerate(folds.split(all_idx, y_all, groups), start=1):
        t0 = time.time()
        fold_train = df.iloc[tr_idx].reset_index(drop=True)
        fold_test = df.iloc[te_idx].reset_index(drop=True)
        assert not set(fold_train.group) & set(fold_test.group)   # no near-duplicate on both sides
        fold_model = train_fresh_model(fold_train, CV_EPOCHS, seed=SEED + fold)
        probs, labels = predict(fold_model, make_loader(fold_test, TRAIN_IMG_DIR, eval_tf, train=False), tta=True)
        fold_loss, fold_acc = score(probs, labels)
        cv_results.append({"fold": fold, "train_photos": len(fold_train), "test_photos": len(fold_test),
                           "accuracy": fold_acc, "loss": fold_loss})
        print(f"Fold {fold}/{CV_FOLDS}: trained on {len(fold_train):,} photos, tested on {len(fold_test):,} -> "
              f"accuracy {fold_acc:.1%}  ({(time.time() - t0) / 60:.1f} min)")
        del fold_model, probs, labels
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()
    print(f"\nCross-validation finished in {(time.time() - cv_start) / 60:.0f} minutes.")
else:
    print("Cross-validation skipped (RUN_CROSS_VALIDATION = False).")

## 7.2 Cross-validation result
- The **mean** is the best estimate of the accuracy on new photos.
- The **standard deviation** shows how much the result depends on which photos happen to be in the test set.
  A small value (around one percentage point) means the result is stable.
- If the main model's test accuracy from Part 5 lies within about two standard deviations of the mean,
  it wasn't a lucky split. Each fold trains for 4 epochs instead of 8, so fold accuracies can be slightly lower.

In [ ]:
if cv_results:
    cv_df = pd.DataFrame(cv_results)
    mean_acc, std_acc = cv_df["accuracy"].mean(), cv_df["accuracy"].std(ddof=1)
    print(f"{CV_FOLDS}-fold cross-validation accuracy: {mean_acc:.1%} ± {std_acc:.1%}  (mean ± standard deviation)")
    print(f"Lowest fold: {cv_df['accuracy'].min():.1%}   Highest fold: {cv_df['accuracy'].max():.1%}")
    print(f"Main model on its own test set (Part 5): {test_acc:.1%}")

    fig, ax = plt.subplots(figsize=(8, 4.5))
    bars = ax.bar(cv_df["fold"].astype(str), cv_df["accuracy"] * 100, color="#4c78a8")
    ax.bar_label(bars, labels=[f"{a:.1%}" for a in cv_df["accuracy"]], padding=3)
    ax.axhline(mean_acc * 100, color="black", linestyle="--", linewidth=1, label=f"mean {mean_acc:.1%}")
    ax.axhspan((mean_acc - std_acc) * 100, (mean_acc + std_acc) * 100, color="gray", alpha=0.25,
               label=f"mean ± 1 standard deviation ({std_acc:.1%})")
    ax.set_ylim(0, 100)
    ax.set_xlabel("Fold (the part of the data used for testing)")
    ax.set_ylabel("Test accuracy (%)")
    ax.set_title(f"{CV_FOLDS}-fold cross-validation: accuracy on each held-out fold")
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()

    cv_df.to_csv(os.path.join(OUTPUT_DIR, "cross_validation.csv"), index=False)
    print("Saved per-fold results to", os.path.join(OUTPUT_DIR, "cross_validation.csv"))
    display(cv_df.round(4))
else:
    print("No cross-validation results (Part 7 was skipped).")

# Summary
| | |
|---|---|
| **Data** | 12,600 labeled photos, 15 actions; near-duplicates grouped; split 80% train / 10% validation / 10% test |
| **Preprocessing** | resize to 224×224, CLIP normalization; augmentation: random crop, flip, rotation, brightness/contrast |
| **Model** | CLIP ViT-B/16 (Vision Transformer, ~86M parameters) pretrained on 400M image–caption pairs, new 15-class head |
| **Training** | full fine-tuning, AdamW, layer-wise learning rates (0.75 decay), warm-up + cosine schedule, label smoothing 0.1, dropout + drop-path 0.1, mixed precision, EMA weights |
| **Evaluation** | test accuracy with flip TTA (section 5.2), per-action scores, confusion matrix |
| **Cross-validation** | 5 group-stratified folds, a fresh model per fold (4 epochs), mean ± standard deviation in section 7.2 |

**Why it beats EfficientNetB0 (76.4%):** CLIP learned from captions that describe what people are *doing*,
the transformer's attention relates distant parts of the photo (hand ↔ phone ↔ ear), and we fine-tuned every layer
instead of only the top.